In [1]:
import os
from pathlib import Path
import pickle
from datasets import load_dataset

curr_dir = Path(os.getcwd())
data_dir = curr_dir / 'data'
if not os.path.exists(data_dir):
    os.mkdir(data_dir)
data_pickle_path = data_dir / 'data_set.pkl'

if not os.path.exists(data_pickle_path):
    print(f"Data set hasn't been loaded. Loading from the datasets library and save it as a pickle.")
    data_set = load_dataset("vipulmaheshwari/GTA-Image-Captioning-Dataset")
    with open(data_pickle_path, 'wb') as outfile:
        pickle.dump(data_set, outfile)
else:
    print(f"Data set already exists in the local drive. Loading it.")
    with open(data_pickle_path, 'rb') as infile:
        data_set = pickle.load(infile)

Data set already exists in the local drive. Loading it.


In [17]:
# print(data_set)
# len(data_set['train']['image']), len(data_set['train']['text'])

In [44]:
# Source: https://huggingface.co/sentence-transformers/clip-ViT-L-14

from sentence_transformers import SentenceTransformer, util
# from PIL import Image

#Load CLIP model
model = SentenceTransformer("sentence-transformers/clip-ViT-L-14") # SentenceTransformer('clip-ViT-L-14')

#Encode an image:
# img_emb = model.encode(image) # Image.open('two_dogs_in_snow.jpg')

# #Encode text descriptions
# text_emb = model.encode(text) # ['Two dogs in the snow', 'A cat on a table', 'A picture of London at night']

# #Compute cosine similarities 
# cos_scores = util.cos_sim(img_emb, text_emb)
# print(cos_scores)

In [ ]:
img_embeddings = []
for image in tqdm(data_set['train']['image'][:2]):
    img_embedding = model.encode(image)
    img_embeddings.append(img_embedding)

# try FAISS. Chroma, Pinecone (check the GAFS project)

In [ ]:
import pyarrow as pa
import lancedb

db = lancedb.connect('./data/tables')
schema = pa.schema(
  [
      pa.field("vector", pa.list_(pa.float32())),
      # pa.field("text", pa.string()),
      # pa.field("id", pa.int32())
  ])
# tbl = db.create_table("gta_data", schema=schema, mode="overwrite")

In [60]:
from tqdm import tqdm
import numpy as np

img_embeddings = []
for image in tqdm(data_set['train']['image'][:2]):
    img_embedding = model.encode(image)
    img_embeddings.append(img_embedding)

tbl_data = pa.Table.from_arrays([pa.array(img_embeddings)], ["vector"])
tbl = db.create_table("gta_data", tbl_data, schema=schema, mode="overwrite")

# tbl.add(img_embeddings)
# tbl.to_pandas()

100%|██████████| 2/2 [00:15<00:00,  7.65s/it]


In [63]:
res = tbl.search(model.encode("a road with a stop"), vector_column_name="vector").limit(3).to_pandas()
res

TypeError: Query column vector must be a vector. Got list<item: float>.

In [ ]:
# https://huggingface.co/openai/clip-vit-large-patch14

In [24]:
import clip
import torch
import os
from datasets import load_dataset

# ds = load_dataset("vipulmaheshwari/GTA-Image-Captioning-Dataset")
# device = torch.device("mps")
model, preprocess = clip.load("ViT-L/14") # , device=device

In [15]:
def embed_txt(txt):
    tokenized_text = clip.tokenize([txt])
    embeddings = model.encode_text(tokenized_text)
    
    # Detach, move to CPU, convert to numpy array, and extract the first element as a list
    result = embeddings.detach().numpy()[0].tolist()
    return result

len(embed_txt("a road with a stop"))

768

In [11]:
# https://vipul-maheshwari.github.io/2024/03/03/multimodal-rag-application

def embed_image(img):
    processed_image = preprocess(img)
    unsqueezed_image = processed_image.unsqueeze(0)
    embeddings = model.encode_image(unsqueezed_image)
    
    # Detach, move to CPU, convert to numpy array, and extract the first element as a list
    result = embeddings.detach().numpy()[0].tolist()
    return result

len(embed_image(image))

[1.172108769416809,
 0.5741956830024719,
 -0.11420677602291107,
 -0.5107784271240234,
 -0.7742195725440979,
 0.7895426750183105,
 0.31811264157295227,
 0.5389135479927063,
 0.17074763774871826,
 -1.0352754592895508,
 -0.013449656777083874,
 -0.5795634388923645,
 -0.37020763754844666,
 -0.7534741163253784,
 0.6788989901542664,
 -0.1245330423116684,
 1.0375893115997314,
 -0.08196641504764557,
 0.169560506939888,
 -0.3306411802768707,
 0.6850194931030273,
 -0.4113234281539917,
 -0.3725243806838989,
 -0.8902166485786438,
 -0.2419223040342331,
 0.33643779158592224,
 0.18724264204502106,
 0.6745221018791199,
 0.00899740681052208,
 -0.29769381880760193,
 0.6830898523330688,
 0.7002785205841064,
 0.5598942041397095,
 -0.27884775400161743,
 0.29804039001464844,
 0.4663200378417969,
 -0.40516427159309387,
 -0.2796509861946106,
 -0.3568377196788788,
 0.7982958555221558,
 1.0218019485473633,
 -0.3191905915737152,
 -0.8690600395202637,
 -0.5986450910568237,
 0.6520456671714783,
 0.8482719659805298,

In [ ]:
def embed_txt(txt):
    tokenized_text = clip.tokenize([txt]).to(device)
    embeddings = model.encode_text(tokenized_text)
    
    # Detach, move to CPU, convert to numpy array, and extract the first element as a list
    result = embeddings.detach().cpu().numpy()[0].tolist()
    return result

res = tbl.search(embed_txt("a road with a stop")).limit(3).to_pandas()
res

In [ ]:
https://blog.lancedb.com/lancedb-polars-2d5eb32a8aa3/

https://github.com/lancedb/lancedb